In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

Note: you may need to restart the kernel to use updated packages.
  Using cached qiskit-aer-0.15.1.tar.gz (6.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build qiskit-aer
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for qiskit-aer (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [113 lines of output]
      C:\Users\jeral\AppData\Local\Temp\pip-build-env-w3q92pmf\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      
      
      --------------------------------------------------------------------------------

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

def quantum_random_bits(n, simulator, max_qubits=24):
    if n <= 0:
        return []
    bits = []
    remaining = n
    while remaining > 0:
        chunk = min(max_qubits, remaining)
        qc = QuantumCircuit(chunk, chunk)
        qc.h(range(chunk))
        qc.measure(range(chunk), range(chunk))
        compiled = transpile(qc, simulator)
        result = simulator.run(compiled, shots=1).result()
        counts = result.get_counts()
        bitstring = next(iter(counts))
        bitstring = bitstring[::-1]  # Align bit 0 with qubit 0.
        bits.extend(int(b) for b in bitstring)
        remaining -= chunk
    return bits


def measure_bit(prep_bit, prep_basis, meas_basis, simulator):
    qc = QuantumCircuit(1, 1)
    if prep_bit == 1:
        qc.x(0)
    if prep_basis == 1:
        qc.h(0)
    if meas_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    counts = result.get_counts()
    measured = int(next(iter(counts)))
    return measured


def sift_key(alice_bits, alice_bases, bob_bits, bob_bases):
    indices = [i for i in range(len(alice_bits)) if alice_bases[i] == bob_bases[i]]
    alice_key = [alice_bits[i] for i in indices]
    bob_key = [bob_bits[i] for i in indices]
    return indices, alice_key, bob_key


def error_rate(alice_key, bob_key):
    if not alice_key:
        return 0.0
    mismatches = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    return mismatches / len(alice_key)


In [3]:
simulator = BasicSimulator()
num_qubits = 64
attack_threshold = 0.15

# Alice: choose random bits and bases, then prepare qubits accordingly.
alice_bits = quantum_random_bits(num_qubits, simulator)
alice_bases = quantum_random_bits(num_qubits, simulator)

# Eve: intercept and measure with her own random bases, then resend.
eve_bases = quantum_random_bits(num_qubits, simulator)
eve_measurements = []
for i in range(num_qubits):
    eve_measurements.append(
        measure_bit(alice_bits[i], alice_bases[i], eve_bases[i], simulator)
    )

# Bob: measure qubits resent by Eve with his own random bases.
bob_bases = quantum_random_bits(num_qubits, simulator)
bob_measurements = []
for i in range(num_qubits):
    bob_measurements.append(
        measure_bit(eve_measurements[i], eve_bases[i], bob_bases[i], simulator)
    )

# Sifting: keep positions where Alice and Bob used the same basis.
sifted_indices, alice_key, bob_key = sift_key(
    alice_bits, alice_bases, bob_measurements, bob_bases
)
err = error_rate(alice_key, bob_key)
discard_rate = 1.0 - (len(alice_key) / num_qubits) if num_qubits > 0 else 0.0
attack_detected = err > attack_threshold

print("BB84 (with intercept-resend attacker)")
print(f"Raw qubits: {num_qubits}")
print(f"Sifted key length: {len(alice_key)}")
print(f"Discard rate: {discard_rate:.2f}")
print(f"Error rate in sifted key: {err:.2f}")
print(f"Attack detected (threshold {attack_threshold:.2f}): {attack_detected}")


BB84 (with intercept-resend attacker)
Raw qubits: 64
Sifted key length: 32
Discard rate: 0.50
Error rate in sifted key: 0.31
Attack detected (threshold 0.15): True


In [4]:
print(f"First 16 sifted key bits (Alice): {alice_key[:16]}")
print(f"First 16 sifted key bits (Bob):   {bob_key[:16]}")


First 16 sifted key bits (Alice): [0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1]
First 16 sifted key bits (Bob):   [0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1]
